In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_aws langchain_core langgraph langgraph-prebuilt langchain-mcp-adapters load_dotenv boto3

import os
from dotenv import load_dotenv
load_dotenv()
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_DEFAULT_REGION"] = os.getenv('AWS_DEFAULT_REGION')

os.environ["LANGSMITH_API_KEY"] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGSMITH_ENDPOINT"]="https://api.smith.langchain.com"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "tutorial"

In [ ]:
import boto3

lambda_client = boto3.client('lambda', region_name='us-east-1')

lambda_function_name = 'get_ticket_details'
result = lambda_client.invoke(
    FunctionName=lambda_function_name,
    InvocationType='RequestResponse',  # Use 'RequestResponse' for synchronous invocation
)

print(result['Payload'].read().decode('utf-8'))

{"ticket_id": "Ticket01", "ticket_description": "Application crashed when executing the import", "failuree_code": "FAIL_03"}


In [ ]:
from langchain_aws import ChatBedrock

def get_ticket_details(ticket_id: str) -> dict:
    """Fetch ticket details for the provided ticket ID."""
    payload = {"ticket_id": ticket_id}
    response = lambda_client.invoke(
        FunctionName="get_ticket_details",
        InvocationType="RequestResponse",
        Payload=json.dumps(payload),
    )
    return json.load(response["Payload"])

def get_failure_details(failure_code: str) -> dict:
    """Get detailed information about a failure using the failure code."""
    payload = {"failure_code": failure_code}
    response = lambda_client.invoke(
        FunctionName="get_failure_details",
        InvocationType="RequestResponse",
        Payload=json.dumps(payload),
    )
    return json.load(response["Payload"])

def fix_failure_steps(failure_code: str) -> dict:
    """Get resolution steps for a given failure code."""
    payload = {"failure_code": failure_code}
    response = lambda_client.invoke(
        FunctionName="fix_failure_steps",
        InvocationType="RequestResponse",
        Payload=json.dumps(payload),
    )
    return json.load(response["Payload"])

def search_failure_bedrock(failure_info: dict) -> dict:
    """Search and explain failure details using Amazon Bedrock (Mistral)."""
    failure_code = failure_info.get("failure_code", "")
    description = failure_info.get("description", "")
    query = f"Explain the failure '{description}' and its cause"

    payload = {"query": query}
    response = lambda_client.invoke(
        FunctionName="search_failure_bedrock",
        InvocationType="RequestResponse",
        Payload=json.dumps(payload),
    )
    return json.load(response["Payload"])

tools = [get_ticket_details, get_failure_details, fix_failure_steps, search_failure_bedrock]
llm = ChatBedrock(model_id="amazon.nova-pro-v1:0", temperature=0.5)
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode
from IPython.display import Image, display
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, SystemMessage

# System message
sys_msg = SystemMessage(content="You are a helpful assistant that can call tools to fetch ticket details, failure information, and resolution steps.")

# Node
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}
# Graph
builder = StateGraph(MessagesState)

# Define nodes: these do the work
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition,
)
builder.add_edge("tools", "assistant")
react_graph = builder.compile()

# Show
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()
react_graph_memory = builder.compile(checkpointer=memory)

In [ ]:
# prompt1: Fetch the details for ticket ID TICKET12345, 
# extract the failure code, use it to retrieve the failure details, 
# and then obtain the appropriate fix or resolution steps for the identified failure.

# prompt2: Fetch the details for ticket ID TICKET12345, 
# extract the failure code, use it to retrieve the failure details, 
# finally search and explain the failure in detail using GenAI.

In [ ]:
def stream_graph_updates(state: MessagesState):
    result = react_graph_memory.invoke(state, config={"configurable": {"thread_id": "1"}})
    print("Assistant:", result["messages"][-1].content)


while True:
    try:
        user_input = input("User: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates({"messages": [{"role": "user", "content": user_input}]})
    except:
        # fallback if input() is not available
        user_input = "find the details for ticket ID TICKET12345, extract the failure code, use it to retrieve the failure details, and then obtain the appropriate fix or resolution steps for the identified failure."
        print("User: " + user_input)
        stream_graph_updates({"messages": [{"role": "user", "content": user_input}]})
        break